# COIN-M Futures Market Stream PoC — `dstream.binance.com`

코인 자체로 증거금·정산을 하는 선물 시장이다. 두 가지 하위 카테고리가 있다.

```text
dstream.binance.com
 ├─ Crypto Perpetual   예: BTCUSD_PERP, ETHUSD_PERP
 └─ Delivery Futures   Current Quarter / Next Quarter
```

**수량 단위가 코인이 아니라 계약 수(contract)다.** `BTCUSD_PERP` 1계약은 100 USD 명목가치이며,
체결 페이로드의 `q`는 계약 수다. 공개 시장 데이터 수신에는 API 키가 필요 없다.

- 관련 문서: [docs/binanceWebSocketREADME.md](../docs/binanceWebSocketREADME.md)


In [ ]:
%pip install -q "websockets>=17,<18"

In [ ]:
import asyncio
import json
import urllib.request

from websockets.asyncio.client import connect

WS_BASE = "wss://dstream.binance.com/ws"
REST_BASE = "https://dapi.binance.com"
SYMBOL = "btcusd_perp"
PAIR = "BTCUSD"

In [ ]:
async def receive(stream: str, message_count: int = 3, timeout: float = 15) -> list[dict]:
    url = f"{WS_BASE}/{stream}"
    messages = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            try:
                message = await asyncio.wait_for(ws.recv(), timeout=timeout)
            except TimeoutError:
                print(f"{timeout}초 동안 추가 메시지 없음: {len(messages)}건 수신")
                break
            messages.append(json.loads(message))

    return messages


def get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=10) as resp:
        return json.load(resp)

## 1. Crypto Perpetual — Trade Stream

`q`(수량)이 코인이 아니라 계약 수임에 유의한다. `BTCUSD_PERP`는 1계약 = 100 USD.

- 스트림: `<symbol>@trade`
- 심볼: `BTCUSD_PERP`


In [ ]:
crypto_perp_trade_messages = await receive(f"{SYMBOL}@trade", message_count=3)
crypto_perp_trade_messages

## 2. Crypto Perpetual — Kline Stream


In [ ]:
KLINE_INTERVAL = "1m"

crypto_perp_kline_messages = await receive(
    f"{SYMBOL}@kline_{KLINE_INTERVAL}",
    message_count=3,
)
crypto_perp_kline_messages

## 3. Crypto Perpetual — Mark Price Stream (마크 가격 · 펀딩비)

USD-M과 동일한 구조다. `r`이 현재 펀딩비율이다.

- 스트림: `<symbol>@markPrice`


In [ ]:
mark_price_messages = await receive(f"{SYMBOL}@markPrice", message_count=3)
mark_price_messages

## 4. Delivery — Current/Next Quarter 심볼 조회

만기 선물 심볼은 만기마다 바뀐다(예: `BTCUSD_260925`). `exchangeInfo`에서 매번 조회한다.
COIN-M의 상태 필드명은 USD-M과 달리 `contractStatus`다.

- REST: `GET /dapi/v1/exchangeInfo`
- 필터: `pair == PAIR` and `contractType in (CURRENT_QUARTER, NEXT_QUARTER)`


In [ ]:
exchange_info = get_json(f"{REST_BASE}/dapi/v1/exchangeInfo")
quarterlies = [
    s for s in exchange_info["symbols"]
    if s.get("pair") == PAIR and s.get("contractType") in ("CURRENT_QUARTER", "NEXT_QUARTER")
]
[(s["symbol"], s["contractType"], s["contractStatus"]) for s in quarterlies]

In [ ]:
current_quarter = next(s["symbol"] for s in quarterlies if s["contractType"] == "CURRENT_QUARTER")
current_quarter

## 5. Delivery — Current Quarter Mark Price Stream

만기 선물은 거래량이 적어 `trade`가 뜸하게 온다. `markPrice`는 체결 여부와 무관하게 갱신되므로
PoC로 더 안정적이다.


In [ ]:
delivery_mark_price_messages = await receive(f"{current_quarter.lower()}@markPrice", message_count=3)
delivery_mark_price_messages

## 6. Combined Streams

Crypto Perpetual의 `trade` + `kline_1m` + `markPrice`를 하나의 연결로 묶어서 수신한다.


In [ ]:
async def receive_combined(streams: list[str], message_count: int = 5) -> list[dict]:
    stream_names = "/".join(streams)
    url = f"wss://dstream.binance.com/stream?streams={stream_names}"
    messages = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            messages.append(json.loads(await ws.recv()))

    return messages


combined_messages = await receive_combined(
    [f"{SYMBOL}@trade", f"{SYMBOL}@kline_1m", f"{SYMBOL}@markPrice"],
    message_count=5,
)
combined_messages